# Credit Card Fraud Detection — Feature Engineering

## Notebook 2: Leakage-Safe Feature Engineering

This notebook transforms the raw transaction data into a structured, model-ready feature set.

The goal is to engineer variables that capture fraud-relevant behavior without introducing data leakage. All train-derived aggregates are learned from the training set only, then applied to the test set.

---

## Objectives

This notebook will:

1. load the predefined training and testing datasets
2. clean and standardize both datasets
3. create time, amount, demographic, geographic, and behavioral features
4. build train-learned aggregation features safely
5. encode and scale the final feature matrix
6. export processed train and test files for modeling

---

## Design Principle

Feature engineering in fraud detection should reflect how fraud actually behaves. Fraud is rarely driven by a single field. Instead, it often emerges through:

- unusual transaction timing
- abnormal spending behavior
- risky merchant or category context
- geographic mismatch
- repeated or burst-like same-day activity

This notebook focuses on translating those patterns into predictive variables.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

In [2]:
OCEAN_PALETTE = [
    "#DFF3F0",
    "#BFEAE5",
    "#79D0D3",
    "#3E9FBF",
    "#1F6F8B",
    "#174A7E",
    "#0F2F5F",
    "#081F3F",
    "#04162E"
]

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

---
## 1. Load the Raw Training and Testing Data

The feature engineering pipeline uses the predefined training and testing files directly. This preserves the intended train/test separation and prevents feature leakage.

In [3]:
TRAIN_PATH = "/kaggle/input/datasets/kaushalnandania/credit-card-fraud-detection/train.csv"
TEST_PATH = "/kaggle/input/datasets/kaushalnandania/credit-card-fraud-detection/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1296675, 23)
Test shape: (555719, 23)


In [4]:
display(train_df.head())
display(test_df.head())

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.9700,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.0113,-82.0483,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.2300,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.1590,-118.1865,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.1100,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.1507,-112.1545,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.0000,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.0343,-112.5611,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.9600,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.6750,-78.6325,0


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.8600,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.9864,-81.2007,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.8400,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.4505,-109.9604,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.2800,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.4958,-74.1961,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.0500,Brian,Williams,M,32941 Krystal Mill Apt. 552,Titusville,FL,32780,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.8124,-80.8831,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.1900,Nathan,Massey,M,5783 Evan Roads Apt. 465,Falmouth,MI,49632,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.9591,-85.8847,0


---
## 2. Basic Structure Check

Before feature engineering, we confirm that the training and testing datasets share the same schema.

In [5]:
print("Columns match:", list(train_df.columns) == list(test_df.columns))

schema_df = pd.DataFrame({
    "column": train_df.columns,
    "train_dtype": train_df.dtypes.astype(str).values,
    "test_dtype": test_df.dtypes.astype(str).values
})
display(schema_df)

Columns match: True


,column,train_dtype,test_dtype
0,Unnamed: 0,int64,int64
1,trans_date_trans_time,object,object
2,cc_num,int64,int64
3,merchant,object,object
4,category,object,object
5,amt,float64,float64
6,first,object,object
7,last,object,object
8,gender,object,object
9,street,object,object


---
## 3. Initial Cleaning

This section applies the same standard cleaning logic to both datasets.

The main tasks are:

- remove index-like unnamed columns if present
- convert dates into proper datetime types
- standardize string fields

In [6]:
def initial_cleaning(df):
    df = df.copy()

    drop_cols = [col for col in df.columns if "unnamed" in col.lower()]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

    object_cols = df.select_dtypes(include="object").columns.tolist()
    for col in object_cols:
        df[col] = df[col].astype(str).str.strip()

    return df

In [7]:
train_df = initial_cleaning(train_df)
test_df = initial_cleaning(test_df)

print("Train shape after cleaning:", train_df.shape)
print("Test shape after cleaning:", test_df.shape)

Train shape after cleaning: (1296675, 22)
Test shape after cleaning: (555719, 22)


---
## 4. Time-Based Features

Fraud risk often changes with transaction timing. We therefore decompose the transaction timestamp into interpretable time features.

In [8]:
def add_time_features(df):
    df = df.copy()

    df["trans_hour"] = df["trans_date_trans_time"].dt.hour
    df["trans_day"] = df["trans_date_trans_time"].dt.day
    df["trans_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek
    df["trans_month"] = df["trans_date_trans_time"].dt.month
    df["trans_year"] = df["trans_date_trans_time"].dt.year
    df["trans_date"] = df["trans_date_trans_time"].dt.date

    df["is_weekend"] = df["trans_dayofweek"].isin([5, 6]).astype(int)
    df["is_night_transaction"] = ((df["trans_hour"] >= 22) | (df["trans_hour"] <= 3)).astype(int)

    return df

In [9]:
train_df = add_time_features(train_df)
test_df = add_time_features(test_df)

display(train_df[[
    "trans_date_trans_time", "trans_hour", "trans_dayofweek",
    "trans_month", "is_weekend", "is_night_transaction"
]].head())

,trans_date_trans_time,trans_hour,trans_dayofweek,trans_month,is_weekend,is_night_transaction
0,2019-01-01 00:00:18,0,1,1,0,1
1,2019-01-01 00:00:44,0,1,1,0,1
2,2019-01-01 00:00:51,0,1,1,0,1
3,2019-01-01 00:01:16,0,1,1,0,1
4,2019-01-01 00:03:06,0,1,1,0,1


---
## 5. Demographic Features

Age is created from birth date and transaction date. Age bands are included because grouped age effects are often more stable than exact-age effects.

In [10]:
def add_demographic_features(df):
    df = df.copy()

    df["age"] = ((df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25).astype(float)
    df["age"] = df["age"].clip(lower=0)

    df["age_band"] = pd.cut(
        df["age"],
        bins=[0, 25, 35, 45, 55, 65, 120],
        labels=["Under 25", "25-34", "35-44", "45-54", "55-64", "65+"],
        right=False
    )

    return df

In [11]:
train_df = add_demographic_features(train_df)
test_df = add_demographic_features(test_df)

display(train_df[["dob", "age", "age_band"]].head())

,dob,age,age_band
0,1988-03-09,30.8145,25-34
1,1978-06-21,40.5311,35-44
2,1962-01-19,56.9500,55-64
3,1967-01-12,51.9699,45-54
4,1986-03-28,32.7639,25-34


---
## 6. Amount Features

Transaction amount is one of the strongest raw fraud signals, but it is highly skewed. This section creates transformed amount features and a high-value indicator derived from the training set.

In [12]:
def add_amount_features(df, high_value_threshold=None):
    df = df.copy()

    df["log_amt"] = np.log1p(df["amt"])
    df["amt_squared"] = df["amt"] ** 2

    if high_value_threshold is None:
        high_value_threshold = df["amt"].quantile(0.95)

    df["is_high_value_txn"] = (df["amt"] >= high_value_threshold).astype(int)

    return df, high_value_threshold

In [13]:
train_df, high_value_threshold = add_amount_features(train_df)
test_df, _ = add_amount_features(test_df, high_value_threshold=high_value_threshold)

print("Train-derived high-value threshold:", round(high_value_threshold, 2))

Train-derived high-value threshold: 196.31


## 7. Population and Band Features

City population is included as a contextual variable. Quantile band definitions are learned from the training data and then applied to the test data.

In [14]:
train_df["log_city_pop"] = np.log1p(train_df["city_pop"])
test_df["log_city_pop"] = np.log1p(test_df["city_pop"])

In [15]:
amt_bins = np.unique(train_df["amt"].quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0]).values)
if len(amt_bins) < 6:
    amt_bins = np.unique(np.linspace(train_df["amt"].min(), train_df["amt"].max(), 6))

city_pop_bins = np.unique(train_df["city_pop"].quantile([0, 0.2, 0.4, 0.6, 0.8, 1.0]).values)
if len(city_pop_bins) < 6:
    city_pop_bins = np.unique(np.linspace(train_df["city_pop"].min(), train_df["city_pop"].max(), 6))

amt_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
city_pop_labels = ["Very Low", "Low", "Medium", "High", "Very High"]

In [16]:
def apply_band_features(df, amt_bins, city_pop_bins):
    df = df.copy()

    df["amt_band"] = pd.cut(
        df["amt"],
        bins=amt_bins,
        labels=amt_labels[:len(amt_bins)-1],
        include_lowest=True,
        duplicates="drop"
    )

    df["city_pop_band"] = pd.cut(
        df["city_pop"],
        bins=city_pop_bins,
        labels=city_pop_labels[:len(city_pop_bins)-1],
        include_lowest=True,
        duplicates="drop"
    )

    return df

In [17]:
train_df = apply_band_features(train_df, amt_bins, city_pop_bins)
test_df = apply_band_features(test_df, amt_bins, city_pop_bins)

display(train_df[["amt", "amt_band", "city_pop", "city_pop_band"]].head())

,amt,amt_band,city_pop,city_pop_band
0,4.9700,Very Low,3495,Medium
1,107.2300,Very High,149,Very Low
2,220.1100,Very High,4154,Medium
3,45.0000,Medium,1939,Medium
4,41.9600,Medium,99,Very Low


---
## 8. Sequential Customer Features

Fraud often appears as a break from a customer's normal behavior. These sequential features are computed separately within train and test so that transaction order is preserved without crossing dataset boundaries.

In [18]:
def add_sequential_customer_features(df):
    df = df.copy()
    df = df.sort_values(["cc_num", "trans_date_trans_time"]).reset_index(drop=True)

    group_cc = df.groupby("cc_num")

    df["prev_amt"] = group_cc["amt"].shift(1)
    df["prev_trans_time"] = group_cc["trans_date_trans_time"].shift(1)

    df["amt_change_from_prev"] = df["amt"] - df["prev_amt"]
    df["amt_ratio_to_prev"] = df["amt"] / (df["prev_amt"] + 1e-6)

    df["time_since_last_txn_sec"] = (
        df["trans_date_trans_time"] - df["prev_trans_time"]
    ).dt.total_seconds()

    df["cust_txn_count"] = group_cc.cumcount() + 1
    df["cust_avg_amt"] = group_cc["amt"].transform("mean")
    df["cust_median_amt"] = group_cc["amt"].transform("median")
    df["cust_std_amt"] = group_cc["amt"].transform("std").fillna(0)

    df["amt_vs_cust_mean"] = df["amt"] - df["cust_avg_amt"]
    df["amt_vs_cust_median"] = df["amt"] - df["cust_median_amt"]

    df["zscore_amt_within_customer"] = np.where(
        df["cust_std_amt"] > 0,
        (df["amt"] - df["cust_avg_amt"]) / df["cust_std_amt"],
        0
    )

    return df

In [19]:
train_df = add_sequential_customer_features(train_df)
test_df = add_sequential_customer_features(test_df)

display(train_df[[
    "cc_num", "amt", "prev_amt", "amt_change_from_prev",
    "amt_ratio_to_prev", "time_since_last_txn_sec", "cust_txn_count"
]].head(10))

,cc_num,amt,prev_amt,amt_change_from_prev,amt_ratio_to_prev,time_since_last_txn_sec,cust_txn_count
0,60416207185,7.2700,NaN,NaN,NaN,NaN,1
1,60416207185,52.9400,7.2700,45.6700,7.2820,"71,862.0000",2
2,60416207185,82.0800,52.9400,29.1400,1.5504,159.0000,3
3,60416207185,34.7900,82.0800,-47.2900,0.4239,"13,838.0000",4
4,60416207185,27.1800,34.7900,-7.6100,0.7813,"1,952.0000",5
5,60416207185,6.8700,27.1800,-20.3100,0.2528,"89,149.0000",6
6,60416207185,8.4300,6.8700,1.5600,1.2271,"11,315.0000",7
7,60416207185,117.1100,8.4300,108.6800,13.8921,"75,285.0000",8
8,60416207185,26.7400,117.1100,-90.3700,0.2283,"26,247.0000",9
9,60416207185,105.2000,26.7400,78.4600,3.9342,"12,302.0000",10


---
## 9. Same-Day Customer Activity Features

These features capture burst-like behavior within the same day, which is often highly relevant in fraud detection.

In [20]:
def add_same_day_features(df):
    df = df.copy()

    cust_day_group = df.groupby(["cc_num", "trans_date"])

    df["customer_txn_same_day"] = cust_day_group["amt"].transform("count")
    df["customer_amt_same_day"] = cust_day_group["amt"].transform("sum")
    df["customer_avg_same_day"] = cust_day_group["amt"].transform("mean")
    df["amt_share_of_day_total"] = df["amt"] / (df["customer_amt_same_day"] + 1e-6)

    return df

In [21]:
train_df = add_same_day_features(train_df)
test_df = add_same_day_features(test_df)

display(train_df[[
    "cc_num", "trans_date", "amt", "customer_txn_same_day",
    "customer_amt_same_day", "customer_avg_same_day", "amt_share_of_day_total"
]].head(10))

,cc_num,trans_date,amt,customer_txn_same_day,customer_amt_same_day,customer_avg_same_day,amt_share_of_day_total
0,60416207185,2019-01-01,7.2700,1,7.2700,7.2700,1.0000
1,60416207185,2019-01-02,52.9400,4,196.9900,49.2475,0.2687
2,60416207185,2019-01-02,82.0800,4,196.9900,49.2475,0.4167
3,60416207185,2019-01-02,34.7900,4,196.9900,49.2475,0.1766
4,60416207185,2019-01-02,27.1800,4,196.9900,49.2475,0.1380
5,60416207185,2019-01-03,6.8700,2,15.3000,7.6500,0.4490
6,60416207185,2019-01-03,8.4300,2,15.3000,7.6500,0.5510
7,60416207185,2019-01-04,117.1100,2,143.8500,71.9250,0.8141
8,60416207185,2019-01-04,26.7400,2,143.8500,71.9250,0.1859
9,60416207185,2019-01-05,105.2000,2,110.1800,55.0900,0.9548


---
## 10. Train-Learned Aggregation Features

This is the most important leakage-safe step in the notebook.

Merchant, category, state, and city summaries are learned from the training data only, then applied to both train and test.

In [22]:
def build_train_mappings(train_df):
    merchant_map = (
        train_df.groupby("merchant")
        .agg(
            merchant_txn_count=("is_fraud", "count"),
            merchant_fraud_rate=("is_fraud", "mean"),
            merchant_avg_amt=("amt", "mean"),
            merchant_median_amt=("amt", "median")
        )
        .reset_index()
    )

    category_map = (
        train_df.groupby("category")
        .agg(
            category_txn_count=("is_fraud", "count"),
            category_fraud_rate=("is_fraud", "mean"),
            category_avg_amt=("amt", "mean"),
            category_median_amt=("amt", "median")
        )
        .reset_index()
    )

    state_map = (
        train_df.groupby("state")
        .agg(
            state_txn_count=("is_fraud", "count"),
            state_fraud_rate=("is_fraud", "mean"),
            state_avg_amt=("amt", "mean"),
            state_median_amt=("amt", "median")
        )
        .reset_index()
    )

    city_map = (
        train_df.groupby("city")
        .agg(
            city_txn_count=("is_fraud", "count"),
            city_fraud_rate=("is_fraud", "mean"),
            city_avg_amt=("amt", "mean"),
            city_median_amt=("amt", "median")
        )
        .reset_index()
    )

    return merchant_map, category_map, state_map, city_map

In [23]:
merchant_map, category_map, state_map, city_map = build_train_mappings(train_df)

In [24]:
global_defaults = {
    "merchant_txn_count": train_df["merchant"].value_counts().mean(),
    "merchant_fraud_rate": train_df["is_fraud"].mean(),
    "merchant_avg_amt": train_df["amt"].mean(),
    "merchant_median_amt": train_df["amt"].median(),

    "category_txn_count": train_df["category"].value_counts().mean(),
    "category_fraud_rate": train_df["is_fraud"].mean(),
    "category_avg_amt": train_df["amt"].mean(),
    "category_median_amt": train_df["amt"].median(),

    "state_txn_count": train_df["state"].value_counts().mean(),
    "state_fraud_rate": train_df["is_fraud"].mean(),
    "state_avg_amt": train_df["amt"].mean(),
    "state_median_amt": train_df["amt"].median(),

    "city_txn_count": train_df["city"].value_counts().mean(),
    "city_fraud_rate": train_df["is_fraud"].mean(),
    "city_avg_amt": train_df["amt"].mean(),
    "city_median_amt": train_df["amt"].median()
}

In [25]:
def apply_train_mappings(df, merchant_map, category_map, state_map, city_map, defaults):
    df = df.copy()

    df = df.merge(merchant_map, on="merchant", how="left")
    df = df.merge(category_map, on="category", how="left")
    df = df.merge(state_map, on="state", how="left")
    df = df.merge(city_map, on="city", how="left")

    for col, val in defaults.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

    return df

In [26]:
train_df = apply_train_mappings(train_df, merchant_map, category_map, state_map, city_map, global_defaults)
test_df = apply_train_mappings(test_df, merchant_map, category_map, state_map, city_map, global_defaults)

display(train_df[[
    "merchant", "merchant_txn_count", "merchant_fraud_rate",
    "category", "category_fraud_rate",
    "state", "state_fraud_rate",
    "city", "city_fraud_rate"
]].head())

,merchant,merchant_txn_count,merchant_fraud_rate,category,category_fraud_rate,state,state_fraud_rate,city,city_fraud_rate
0,"fraud_Jones, Sawayn and Romaguera",1269,0.0095,misc_net,0.0145,WY,0.0057,Fort Washakie,0.0060
1,fraud_Berge LLC,2677,0.0034,gas_transport,0.0047,WY,0.0057,Fort Washakie,0.0060
2,fraud_Luettgen PLC,2630,0.0046,gas_transport,0.0047,WY,0.0057,Fort Washakie,0.0060
3,fraud_Daugherty LLC,2230,0.0036,kids_pets,0.0021,WY,0.0057,Fort Washakie,0.0060
4,fraud_Beier and Sons,2473,0.0004,home,0.0016,WY,0.0057,Fort Washakie,0.0060


---
## 11. Interaction Features

Fraud often emerges from combinations of conditions rather than individual values. This section creates interaction variables linking amount, timing, and contextual risk.

In [27]:
def add_interaction_features(df):
    df = df.copy()

    df["merchant_risk_amt"] = df["amt"] * df["merchant_fraud_rate"]
    df["category_risk_amt"] = df["amt"] * df["category_fraud_rate"]
    df["state_risk_amt"] = df["amt"] * df["state_fraud_rate"]

    df["night_amt"] = df["amt"] * df["is_night_transaction"]
    df["night_log_amt"] = df["log_amt"] * df["is_night_transaction"]

    df["night_merchant_risk"] = df["merchant_fraud_rate"] * df["is_night_transaction"]
    df["night_category_risk"] = df["category_fraud_rate"] * df["is_night_transaction"]

    return df

In [28]:
train_df = add_interaction_features(train_df)
test_df = add_interaction_features(test_df)

display(train_df[[
    "amt", "is_night_transaction", "night_amt", "night_log_amt",
    "merchant_risk_amt", "category_risk_amt", "state_risk_amt"
]].head())

,amt,is_night_transaction,night_amt,night_log_amt,merchant_risk_amt,category_risk_amt,state_risk_amt
0,7.2700,0,0.0000,0.0000,0.0687,0.1051,0.0414
1,52.9400,0,0.0000,0.0000,0.1780,0.2485,0.3014
2,82.0800,0,0.0000,0.0000,0.3745,0.3853,0.4673
3,34.7900,0,0.0000,0.0000,0.1248,0.0736,0.1981
4,27.1800,0,0.0000,0.0000,0.0110,0.0437,0.1547


---
## 12. Distance Feature

A simple customer-to-merchant distance proxy is created from customer and merchant coordinates.

In [29]:
def add_distance_feature(df):
    df = df.copy()

    df["distance_cust_merchant"] = np.sqrt(
        (df["lat"] - df["merch_lat"]) ** 2 + (df["long"] - df["merch_long"]) ** 2
    )

    return df

In [30]:
train_df = add_distance_feature(train_df)
test_df = add_distance_feature(test_df)

display(train_df[["lat", "long", "merch_lat", "merch_long", "distance_cust_merchant"]].head())

,lat,long,merch_lat,merch_long,distance_cust_merchant
0,43.0048,-108.8964,43.9747,-109.7419,1.2867
1,43.0048,-108.8964,42.0188,-109.0442,0.9970
2,43.0048,-108.8964,42.9613,-109.1576,0.2648
3,43.0048,-108.8964,42.2282,-108.7477,0.7907
4,43.0048,-108.8964,43.3217,-108.0911,0.8654


---
## 13. Handle Missing Values Introduced by Engineering

Sequential features naturally create missing values for first-observed transactions. These are filled in a model-safe manner.

In [31]:
fill_zero_cols = [
    "prev_amt",
    "amt_change_from_prev",
    "amt_ratio_to_prev",
    "time_since_last_txn_sec",
    "amt_vs_cust_mean",
    "amt_vs_cust_median",
    "zscore_amt_within_customer",
    "distance_cust_merchant",
    "night_amt",
    "night_log_amt",
    "night_merchant_risk",
    "night_category_risk",
    "merchant_risk_amt",
    "category_risk_amt",
    "state_risk_amt"
]

band_cols = ["age_band", "amt_band", "city_pop_band"]

In [32]:
def finalize_missing_values(df, fill_zero_cols, band_cols):
    df = df.copy()

    for col in fill_zero_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    for col in band_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).fillna("Unknown")

    return df

In [33]:
train_df = finalize_missing_values(train_df, fill_zero_cols, band_cols)
test_df = finalize_missing_values(test_df, fill_zero_cols, band_cols)

print("Remaining missing values in train:", train_df.isna().sum().sum())
print("Remaining missing values in test:", test_df.isna().sum().sum())

Remaining missing values in train: 983
Remaining missing values in test: 924


---
## 14. Remove Identifiers and Non-Model Columns

The feature matrix should not include identifiers, descriptive text fields, or raw fields that are unsuitable for modeling.

In [34]:
drop_model_cols = [
    "trans_date_trans_time",
    "trans_date",
    "dob",
    "first",
    "last",
    "street",
    "city",
    "state",
    "zip",
    "job",
    "merchant",
    "trans_num",
    "unix_time",
    "cc_num",
    "prev_trans_time"
]

existing_drop_train = [col for col in drop_model_cols if col in train_df.columns]
existing_drop_test = [col for col in drop_model_cols if col in test_df.columns]

train_model = train_df.drop(columns=existing_drop_train).copy()
test_model = test_df.drop(columns=existing_drop_test).copy()

print("Train modeling shape:", train_model.shape)
print("Test modeling shape:", test_model.shape)

Train modeling shape: (1296675, 63)
Test modeling shape: (555719, 63)


---
## 15. Split Features and Target

In [35]:
target_col = "is_fraud"

X_train = train_model.drop(columns=[target_col]).copy()
y_train = train_model[target_col].copy()

X_test = test_model.drop(columns=[target_col]).copy()
y_test = test_model[target_col].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1296675, 62)
X_test shape: (555719, 62)
y_train shape: (1296675,)
y_test shape: (555719,)


---
## 16. One-Hot Encode Categorical Variables

Categorical variables are encoded using the training schema and then aligned with the test set.

In [36]:
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['category', 'gender', 'age_band', 'amt_band', 'city_pop_band']


In [37]:
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=False)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=False)

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

print("Encoded train shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

Encoded train shape: (1296675, 89)
Encoded test shape: (555719, 89)


---
## 17. Scale Numerical Features

Scaling is fitted on the training set and applied to the test set. Binary indicator columns are left unchanged.

In [38]:
numeric_cols = X_train_encoded.select_dtypes(include=[np.number]).columns.tolist()
binary_like_cols = [col for col in numeric_cols if X_train_encoded[col].dropna().isin([0, 1]).all()]
scale_cols = [col for col in numeric_cols if col not in binary_like_cols]

print("Numeric columns:", len(numeric_cols))
print("Binary-like columns:", len(binary_like_cols))
print("Columns to scale:", len(scale_cols))

Numeric columns: 57
Binary-like columns: 3
Columns to scale: 54


In [39]:
scaler = StandardScaler()

X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train_scaled[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test_scaled[scale_cols])

---
## 18. Final Dataset Checks

In [40]:
print("Final train feature shape:", X_train_scaled.shape)
print("Final test feature shape:", X_test_scaled.shape)
print("Missing values in X_train_scaled:", X_train_scaled.isna().sum().sum())
print("Missing values in X_test_scaled:", X_test_scaled.isna().sum().sum())
print("Duplicate columns in X_train_scaled:", X_train_scaled.columns.duplicated().sum())

Final train feature shape: (1296675, 89)
Final test feature shape: (555719, 89)
Missing values in X_train_scaled: 0
Missing values in X_test_scaled: 0
Duplicate columns in X_train_scaled: 0


In [41]:
preview_cols = [col for col in [
    "amt",
    "log_amt",
    "trans_hour",
    "trans_month",
    "is_night_transaction",
    "age",
    "log_city_pop",
    "prev_amt",
    "amt_change_from_prev",
    "amt_ratio_to_prev",
    "time_since_last_txn_sec",
    "cust_txn_count",
    "cust_avg_amt",
    "customer_txn_same_day",
    "customer_amt_same_day",
    "merchant_fraud_rate",
    "category_fraud_rate",
    "state_fraud_rate",
    "city_fraud_rate",
    "night_amt",
    "distance_cust_merchant"
] if col in X_train_scaled.columns]

display(X_train_scaled[preview_cols].head())
display(X_test_scaled[preview_cols].head())

,amt,log_amt,trans_hour,trans_month,is_night_transaction,age,log_city_pop,prev_amt,amt_change_from_prev,amt_ratio_to_prev,time_since_last_txn_sec,cust_txn_count,cust_avg_amt,customer_txn_same_day,customer_amt_same_day,merchant_fraud_rate,category_fraud_rate,state_fraud_rate,city_fraud_rate,night_amt,distance_cust_merchant
0,-0.3935,-1.1019,-0.1181,-1.5046,0,-0.7546,-0.3898,-0.4386,-0.0001,-0.1824,-0.6849,-1.3409,-0.7381,-1.2512,-0.7207,0.6613,1.6158,-0.0332,0.0077,-0.2133,1.8298
1,-0.1086,0.3524,-0.7047,-1.5046,0,-0.7545,-0.3898,-0.3932,0.2077,0.0556,0.8314,-1.3394,-0.7381,-0.3124,-0.3271,-0.4376,-0.2040,-0.0332,0.0077,-0.2133,0.8126
2,0.0732,0.6874,-0.7047,-1.5046,0,-0.7545,-0.3898,-0.1081,0.1325,-0.1318,-0.6816,-1.3379,-0.7381,-0.3124,-0.3271,-0.2211,-0.2040,-0.0332,0.0077,-0.2133,-1.7591
3,-0.2218,0.0343,-0.1181,-1.5046,0,-0.7545,-0.3898,0.0738,-0.2153,-0.1686,-0.3929,-1.3365,-0.7381,-0.3124,-0.3271,-0.3969,-0.6848,-0.0332,0.0077,-0.2133,0.0879
4,-0.2693,-0.1511,0.0286,-1.5046,0,-0.7545,-0.3898,-0.2214,-0.0347,-0.1569,-0.6438,-1.3350,-0.7381,-0.3124,-0.3271,-0.9709,-0.7791,-0.0332,0.0077,-0.2133,0.3502


,amt,log_amt,trans_hour,trans_month,is_night_transaction,age,log_city_pop,prev_amt,amt_change_from_prev,amt_ratio_to_prev,time_since_last_txn_sec,cust_txn_count,cust_avg_amt,customer_txn_same_day,customer_amt_same_day,merchant_fraud_rate,category_fraud_rate,state_fraud_rate,city_fraud_rate,night_amt,distance_cust_merchant
0,0.3388,1.0083,0.0286,-0.0416,0,-0.6701,-0.3898,-0.4386,-0.0001,-0.1824,-0.6849,-1.3409,-0.1984,-0.9382,-0.3143,-0.7421,-0.7791,-0.0332,0.0077,-0.2133,-1.3885
1,0.0510,0.6534,0.4686,-0.0416,0,-0.6701,-0.3898,0.3396,-0.2101,-0.1619,-0.4319,-1.3394,-0.1984,-0.9382,-0.3143,-0.3663,-0.4939,-0.0332,0.0077,-0.2133,0.2593
2,-0.0318,0.5118,-0.8514,-0.0416,0,-0.6699,-0.3898,0.0516,-0.0605,-0.1553,0.4962,-1.3379,-0.1984,-0.9382,-0.4184,-0.0047,-0.2040,-0.0332,0.0077,-0.2133,1.6215
3,0.1085,0.7385,0.3220,-0.0416,0,-0.6699,-0.3898,-0.0313,0.1022,-0.1385,-0.1102,-1.3365,-0.1984,-0.9382,-0.4184,-0.3206,-0.6848,-0.0332,0.0077,-0.2133,-0.4869
4,0.4845,1.1405,-0.1181,-0.0416,0,-0.6698,-0.3898,0.1091,0.2742,-0.1273,0.9057,-1.3350,-0.1984,0.0006,-0.3838,-0.3619,-0.6271,-0.0332,0.0077,-0.2133,-1.0901


---
## 19. Engineered Feature Families

The final feature set includes the following major groups:

1. Time-based features  
   - transaction hour
   - day of week
   - month
   - weekend indicator
   - night indicator

2. Demographic features  
   - age
   - age band

3. Amount features  
   - raw amount
   - log amount
   - squared amount
   - high-value transaction indicator
   - amount band

4. Sequential customer behavior  
   - previous amount
   - amount change
   - ratio to previous
   - time since last transaction
   - running customer count
   - customer-centered deviation measures

5. Same-day activity  
   - same-day transaction count
   - same-day amount total
   - same-day average amount
   - share of same-day spend

6. Train-learned aggregation features  
   - merchant fraud rate
   - category fraud rate
   - state fraud rate
   - city fraud rate
   - related activity and amount summaries

7. Interaction features  
   - night amount
   - risk-weighted amount
   - night risk interactions

8. Geographic context  
   - city population
   - city population band
   - customer-to-merchant distance

---
## 20. Export Processed Files

The final processed train and test feature sets are saved for the modeling notebook.

In [42]:
train_processed = X_train_scaled.copy()
train_processed[target_col] = y_train.values

test_processed = X_test_scaled.copy()
test_processed[target_col] = y_test.values

train_processed.to_parquet("fraud_train_features.parquet", index=False)
test_processed.to_parquet("fraud_test_features.parquet", index=False)

train_processed.to_csv("fraud_train_features.csv", index=False)
test_processed.to_csv("fraud_test_features.csv", index=False)

print("Saved files:")
print("- fraud_train_features.parquet")
print("- fraud_test_features.parquet")
print("- fraud_train_features.csv")
print("- fraud_test_features.csv")

Saved files:
- fraud_train_features.parquet
- fraud_test_features.parquet
- fraud_train_features.csv
- fraud_test_features.csv


---
# Final Conclusion

Notebook 2 established the feature engineering backbone of the fraud detection pipeline.

The core result of this notebook is a train/test-safe transformation of raw transaction data into a richer behavioral and contextual representation of fraud risk. Rather than relying on raw columns alone, the engineered dataset now reflects:

- transaction intensity
- anomaly relative to prior behavior
- time-based exposure
- location and merchant context
- same-day concentration of activity

This engineered feature set will serve as the input to the next notebook, where baseline and optimized machine learning models will be trained and compared.